# Building the Transition Matrix in Python

### FSRM Academy · Coding Series · Transition Matrix

---

This is a walkthrough of the **Transition Matrix** in Python — the same process
you know from the working paper, built step by step from the raw transactional
data. Follow along and run each cell yourself.

This session covers the **Transition Matrix only**. The parts that follow it —
net flow rate, ECL, and so on — are separate sessions.

> **Where we're headed:** for a loan in each aging bucket today, its probability of
> default within 12 months — the *Through-the-Cycle PD*.

### The steps

```
   Raw transactional data (5 years of monthly extracts)
       │
       ▼
   1. Consolidate        stack every monthly file into one dataset
   2. Pivot              one row per loan, report months across, DPD in cells
   3. Bucket             DPD  ->  aging bucket 0–4
   4. Everbad            once Bad (4), always Bad
   5. Default rate (ODR) good → bad over a 12-month window
   6. Build the matrix   Count  ->  Percentage  ->  Average
   7. Markov             multiply the average matrix by itself 12 times
       │
       ├──►  8. TTC PD               last column of the 12-step matrix  ← the output
       └──►  9. Compare vs actual    model number vs the observed rate
```

Everything today is done with **pandas** — no other library. We assume the pandas
basics (DataFrames, filtering, `groupby`) are already covered, and each new bit of
code gets a quick explanation the first time it shows up.

**One rule for today:** after every cell, look at the **shape** of what came back.
`df.shape` is the fastest way to catch a mistake.

> **▶ FACILITATOR — quick reminders** (full talk track: `FACILITATOR_SCRIPT.md`)
>
> - You're on the **tech side** — teach the **code**, hand risk questions back to
>   the room. This session is the **Transition Matrix** only (net flow rate, ECL are
>   colleagues' sessions).
> - The rhythm is: **say what we're about to do → run the cell → show the result.**
> - **Say the shape** after each cell — that's what keeps people with you.
> - **Numbers:** sample data, so the PD is ~**7.6%**, not the ~**11.64%** on the
>   real portfolio. Say so up front.
> - **Timing:** ~65–70 min. The consolidation read cell is the slow one (~40s) —
>   start it, talk over it. Before you start: Kernel → Restart, run Setup.
> - The grey boxes are only in this copy (the audience's is clean).

---
## Setup

Two small steps: bring in the library, then tell Python where the data is.

**Step 1 — bring in the library we'll use.** Everything today is **pandas** — the
one you learned this morning. It handles the data, the maths, the tables, and even
the charts. (`Path` is from Python itself, just to find the files.)

In [1]:
import pandas as pd
from pathlib import Path

**Step 2 — point Python at the data folder.**

In [2]:
# transition_matrix.py sits next to this notebook. We borrow one helper from it
# that finds the data folder (and builds a sample set the first time, if needed).
import transition_matrix as tm

DATA_ROOT = tm.find_data_root()
print("Data folder:", DATA_ROOT)

Data folder: /Users/cherylannecalma/Documents/Local Repos/ecl-v2/transition_matrix_presentation/data/Transactional Data


---
## The input data

Our starting point is **5 years of monthly transactional extracts** — one file per
month, 60 in all, each a month-end snapshot of the loan book. We assume they're
already cleaned and consistently named (same columns each month, correct data
types).

Only four columns do the real work downstream: **`pn_number`**, **`loan_segment`**,
**`report_date`**, **`dpd`**. Let's open one file and look at it.

In [3]:
# Peek at one raw monthly extract — this is what a single snapshot looks like.
one_file = sorted(DATA_ROOT.glob("*/Transactional Report *.xlsx"))[0]
sample = pd.read_excel(one_file)
print(one_file.name, "->", sample.shape)
sample.head()

Transactional Report April 2020.xlsx -> (923, 16)


,report_date,loan_segment,cis_number,pn_number,loan_amount,outstanding_balance,amortization,booking_date,maturity_date,eir,monthly_payments,dpd,economic_activity,branch,monthly_income_range,age
0,2020-04-30,Salary Loan,BR183,PN2,100000,54043.64,2178.33,2018-01-31,2023-01-31,0.11082,3666.16,0,Accommodation & Food Service,Quezon City,"50,000 - 99,999",23
1,2020-04-30,Consumer Loan,BR150,PN4,120000,113935.42,2624.98,2020-01-31,2025-01-31,0.11265,0.00,0,Real Estate,Davao,"50,000 - 99,999",27
2,2020-04-30,Salary Loan,BR1184,PN12,100000,23474.45,2470.49,2016-06-30,2021-06-30,0.16725,0.00,0,Wholesale & Retail Trade,Baguio,"30,000 - 49,999",43
3,2020-04-30,Salary Loan,BR786,PN14,150000,81201.21,4437.09,2017-02-28,2024-02-28,0.31459,0.00,0,Construction,Makati,"30,000 - 49,999",39
4,2020-04-30,Salary Loan,BR722,PN17,100000,4553.12,5023.73,2018-06-30,2020-06-30,0.18647,0.00,0,Manufacturing,Cagayan de Oro,"Below 15,000",42


---
## 1 · Consolidate the monthly files

First we bring all 60 monthly files together into one dataset. Three lines: read
each Excel file into a **DataFrame** (the table you met in the pandas session),
collect them in a list, and stack the whole list into one table with `concat`.

> 💡 One habit: read every file into the list first, then `concat` once at the end.
> Stacking *inside* the loop recopies the whole table each pass and gets slow.

**Step 1 — list the input files.**

In [4]:
files = sorted(DATA_ROOT.glob("*/Transactional Report *.xlsx"))
print(f"Found {len(files)} monthly files")

Found 60 monthly files


**Step 2 — read each file into a DataFrame.** (This is the slow one — ~40s.)

In [5]:
frames = [pd.read_excel(f) for f in files]   # a list of ~60 tables
print("One file :", frames[0].shape)         # (rows, columns)

One file : (923, 16)


**Step 3 — stack them all into one table** with `concat`.

In [6]:
panel = pd.concat(frames, ignore_index=True)
print("Consolidated:", panel.shape, " <- about 60x the rows")
panel.head()

Consolidated: (57967, 16)  <- about 60x the rows


,report_date,loan_segment,cis_number,pn_number,loan_amount,outstanding_balance,amortization,booking_date,maturity_date,eir,monthly_payments,dpd,economic_activity,branch,monthly_income_range,age
0,2020-04-30,Salary Loan,BR183,PN2,100000,54043.64,2178.33,2018-01-31,2023-01-31,0.11082,3666.16,0,Accommodation & Food Service,Quezon City,"50,000 - 99,999",23
1,2020-04-30,Consumer Loan,BR150,PN4,120000,113935.42,2624.98,2020-01-31,2025-01-31,0.11265,0.00,0,Real Estate,Davao,"50,000 - 99,999",27
2,2020-04-30,Salary Loan,BR1184,PN12,100000,23474.45,2470.49,2016-06-30,2021-06-30,0.16725,0.00,0,Wholesale & Retail Trade,Baguio,"30,000 - 49,999",43
3,2020-04-30,Salary Loan,BR786,PN14,150000,81201.21,4437.09,2017-02-28,2024-02-28,0.31459,0.00,0,Construction,Makati,"30,000 - 49,999",39
4,2020-04-30,Salary Loan,BR722,PN17,100000,4553.12,5023.73,2018-06-30,2020-06-30,0.18647,0.00,0,Manufacturing,Cagayan de Oro,"Below 15,000",42


> **▶ DO:** Step 2 (the read) takes ~40s — start it, then keep talking. When it's
> done: "one file's about a thousand rows; stacked, about sixty thousand — 60 times
> a thousand, so nothing dropped."
>
> **▶ SAY:** "`ignore_index=True` just renumbers the rows cleanly."

---
## Data validation

We're *assuming* the data is clean — but before we build anything on it, it's good
practice to actually check. (This is the same kind of check the ECL app runs on any
file before it uses it.) Four quick checks on our consolidated data:

- the **required columns** are all present
- there are **no duplicate columns**
- **`report_date`** really reads as a date, and **`dpd`** as a number
- no column is **more than 10% missing**

**Step 1 — write the checks.**

In [7]:
REQUIRED = ["pn_number", "loan_segment", "report_date", "dpd"]

def validate_transactional_data(df):
    issues = []
    cols = {str(c).strip().lower(): c for c in df.columns}   # case-insensitive lookup

    # 1) required columns present
    missing_cols = [c for c in REQUIRED if c not in cols]
    if missing_cols:
        issues.append(f"Missing required columns: {missing_cols}")

    # 2) no duplicate columns
    dupes = df.columns[df.columns.duplicated()].tolist()
    if dupes:
        issues.append(f"Duplicate columns: {dupes}")

    # 3) report_date is a real date, dpd is a real number
    if "report_date" in cols:
        parsed = pd.to_datetime(df[cols["report_date"]], errors="coerce")
        bad = int((parsed.isna() & df[cols["report_date"]].notna()).sum())
        if bad:
            issues.append(f"report_date: {bad} values are not valid dates")
    if "dpd" in cols:
        num = pd.to_numeric(df[cols["dpd"]], errors="coerce")
        bad = int((num.isna() & df[cols["dpd"]].notna()).sum())
        if bad:
            issues.append(f"dpd: {bad} values are not numeric")

    # 4) missing values per column — flag anything over 10% (a common threshold)
    n = len(df)
    for c in df.columns:
        miss = int(df[c].isna().sum())
        if miss and miss / n > 0.10:
            issues.append(f"{c}: {miss} missing ({miss/n:.1%}) — over 10%")

    return {"status": "passed" if not issues else "failed", "issues": issues}

**Step 2 — run the checks on our consolidated data.**

In [8]:
result = validate_transactional_data(panel)

print("Validation:", result["status"].upper())
for msg in result["issues"]:
    print("  -", msg)
if result["status"] == "passed":
    print("  Required columns present, no duplicates, report_date & dpd the right")
    print("  type, nothing over 10% missing — safe to build on.")

Validation: PASSED
  Required columns present, no duplicates, report_date & dpd the right
  type, nothing over 10% missing — safe to build on.


**Step 3 — it passed, so set the two key columns to their working types** and we're
ready to reshape.

In [9]:
panel["report_date"] = pd.to_datetime(panel["report_date"])
panel["dpd"] = pd.to_numeric(panel["dpd"], errors="coerce")

print("Distinct loans      :", f"{panel['pn_number'].nunique():,}")
print("Distinct month-ends :", panel["report_date"].nunique())

Distinct loans      : 2,084
Distinct month-ends : 60


> **▶ SAY:** "We're assuming the data's clean, but it's worth actually checking
> before we build on it. Same checks either way: right columns, no duplicates, dates
> and DPD the right type, nothing over 10% blank. Ours passes."
>
> **▶ If asked:** on a failure it lists the problems, and you'd fix the data before
> going on — same messages this would print.

---
## 2 · Pivot to a loan × month grid

Right now the data is **long** — one row per loan *per month*. To see a loan
*move*, we reshape it **wide**: one row per loan, one column per month, DPD in the
cells. pandas does this in one call, `pivot_table`, which asks four things: what's
each **row** (`index`), each **column** (`columns`), what fills the **cells**
(`values`), and what to do with duplicates (`aggfunc`).

Why bother? Because once it's a grid, a loan's **movement** is just two cells side
by side in the same row — which is the whole thing we're about to count.

**Step 1 — pivot the long table into the wide grid.**

In [10]:
wide = panel.pivot_table(
    index="pn_number",       # unique PN numbers -> rows
    columns="report_date",   # report months     -> columns
    values="dpd",            # DPD               -> cells
    aggfunc="first",
)
print("Long :", panel.shape)
print("Wide :", wide.shape, " <- one row per loan now")
wide.iloc[:6, :6]

Long : (57967, 16)
Wide : (2084, 60)  <- one row per loan now


report_date,2020-01-31,2020-02-29,2020-03-31,2020-04-30,2020-05-31,2020-06-30
pn_number,,,,,,
PN1,NaN,NaN,NaN,NaN,NaN,NaN
PN100,NaN,NaN,NaN,NaN,NaN,NaN
PN1000,0.0,0.0,0.0,0.0,0.0,0.0
PN1001,NaN,NaN,NaN,NaN,NaN,NaN
PN1002,NaN,NaN,NaN,NaN,NaN,NaN
PN1003,0.0,0.0,0.0,0.0,0.0,0.0


**Step 2 — tidy the month labels and put them in order.**

In [11]:
wide.columns = [c.strftime("%Y-%m-%d") for c in wide.columns]
months = sorted(wide.columns)
wide = wide[months]
wide.iloc[:6, :6]

,2020-01-31,2020-02-29,2020-03-31,2020-04-30,2020-05-31,2020-06-30
pn_number,,,,,,
PN1,NaN,NaN,NaN,NaN,NaN,NaN
PN100,NaN,NaN,NaN,NaN,NaN,NaN
PN1000,0.0,0.0,0.0,0.0,0.0,0.0
PN1001,NaN,NaN,NaN,NaN,NaN,NaN
PN1002,NaN,NaN,NaN,NaN,NaN,NaN
PN1003,0.0,0.0,0.0,0.0,0.0,0.0


The blanks (`NaN`) are **meaningful**: the account wasn't on the book that month —
not yet drawn, or already closed. Later, when we count movements, these become the
**Indeterminate** group.

In [12]:
filled = wide.notna().sum().sum()
total = wide.shape[0] * wide.shape[1]
print(f"Grid          : {wide.shape[0]:,} loans x {wide.shape[1]} months")
print(f"On book (filled): {filled:,} ({filled/total:.0%})")
print(f"Not on book (NA): {total-filled:,} ({1-filled/total:.0%})")

Grid          : 2,084 loans x 60 months
On book (filled): 57,967 (46%)
Not on book (NA): 67,073 (54%)


---
## 3 · Bucket the DPD

Next we turn the raw day count into an **aging bucket**, 0 to 4, using the standard
bands:

| DPD | Bucket |
|---|---|
| 0 | 0 |
| 1 – 30 | 1 |
| 31 – 60 | 2 |
| 61 – 90 | 3 |
| 91+ | 4 · **default** |

*(The bands can vary by institution policy — set them to yours.)* We write the rule
once as a small **function** and let pandas apply it to every cell.

**Write the rule as a function.** A couple of things to note as we read it:

- `pd.isna(v)` tests for a blank. Missing must **stay** missing — it returns
  `float("nan")`, *not* 0. (Not-on-book is not the same as paying perfectly.)
- The **order** of the `if`s matters — each one only sees what got past the ones
  above it.

In [13]:
def bucket(v):
    if pd.isna(v):
        return float("nan")     # missing stays missing
    if v == 0:
        return 0
    if v <= 30:
        return 1
    if v <= 60:
        return 2
    if v <= 90:
        return 3
    return 4

# Test — all five should print True
print(bucket(0) == 0, bucket(15) == 1, bucket(45) == 2,
      bucket(75) == 3, bucket(120) == 4)

True True True True True


> **▶ SAY:** "Order matters because each check only sees what got past the one
> above it. Put the 90 check first and everything lands in the wrong bucket — and it
> won't error, it'll just be wrong, which is worse."

In [14]:
# Apply the rule to every cell of the grid. (.apply runs the function per value.)
buckets_raw = wide.copy()
for m in months:
    buckets_raw[m] = buckets_raw[m].apply(bucket)

buckets_raw.iloc[:6, :6]

,2020-01-31,2020-02-29,2020-03-31,2020-04-30,2020-05-31,2020-06-30
pn_number,,,,,,
PN1,NaN,NaN,NaN,NaN,NaN,NaN
PN100,NaN,NaN,NaN,NaN,NaN,NaN
PN1000,0.0,0.0,0.0,0.0,0.0,0.0
PN1001,NaN,NaN,NaN,NaN,NaN,NaN
PN1002,NaN,NaN,NaN,NaN,NaN,NaN
PN1003,0.0,0.0,0.0,0.0,0.0,0.0


---
## 4 · Apply the everbad rule

Good = buckets **0–3**, Bad = bucket **4**. In the raw data, loans **recover** from
default. The everbad rule ignores those recoveries: once an account has ever hit
bucket 4, it stays 4 for the rest of its life. Default becomes a one-way door.

We do it with one small **loop** that walks along each account's row, left to right,
remembering "have we seen a default yet?" and setting everything after the first 4
to 4.

In [15]:
buckets = buckets_raw.copy()

for idx in buckets.index:                 # each account
    seen_default = False
    for m in months:                      # each month, left to right
        v = buckets.at[idx, m]
        if pd.isna(v):
            continue                      # not on the book — skip, don't reset
        if seen_default:
            buckets.at[idx, m] = 4        # force it Bad
        elif v == 4:
            seen_default = True           # from here on, always Bad

print("Everbad applied.")

Everbad applied.


A loop is the right tool here: each cell depends on the cells before it, and we
carry a memory (`seen_default`) along the row. How much did the rule change?

In [16]:
raw = buckets_raw[months]
fixed = buckets[months]

on_book = raw.notna()                                   # cells that had a value
changed = int((on_book & (raw != fixed)).sum().sum())   # how many we rewrote

was_bad = (raw == 4)                                    # True where raw = default
ever_bad = int(was_bad.any(axis=1).sum())               # accounts ever bad
# .cummax across the row is True from the first default onward:
after_first_bad = was_bad.cummax(axis=1)
recovered = (after_first_bad & on_book & (raw != 4)).any(axis=1)
cured = int(recovered.sum())

print(f"Accounts that ever defaulted : {ever_bad:,}")
print(f"  …that later recovered      : {cured:,}")
print(f"Cells rewritten to Bad       : {changed:,}  ({changed/int(on_book.sum().sum()):.1%} of all data)")

Accounts that ever defaulted : 387
  …that later recovered      : 334
Cells rewritten to Bad       : 5,768  (10.0% of all data)


> **▶ SAY:** "It walks left to right; the moment it hits a 4, it sets everything
> after to 4. The impact cell just shows how many that changed."
> **▶ If asked why:** hand it to the room — "you'll know the reasoning better than
> me."

---
## 5 · Observed default rate (ODR)

The **observed** default rate — measured straight from the data, no matrix. Of the
Good accounts at a point in time, how many turned Bad over the next 12 months,
divided by the total Good at the start. We use it twice: as a reality check on the
model later, and as the historical series at the end.

The trick in Python is a **boolean mask**: comparing a whole column (like
`buckets == 4`) gives a True/False for every row. `"Good at the start"` and
`"Bad 12 months later"` are two of those; combine them, count, divide.

**Let's do one first**, by hand — observation month **Jan 2021**, a 12-month window
ending **Jan 2022**. Good at the start (`.isin([0,1,2,3])`), of those how many are
Bad twelve months later, then divide.

In [17]:
obs = "2021-01-31"
end = "2022-01-31"

good = buckets[obs].isin([0, 1, 2, 3])       # Good at the observation point
turned_bad = good & (buckets[end] == 4)       # Bad 12 months later

print(f"1. Good in {obs}          : {good.sum():,}")
print(f"2. …that turned Bad by {end} : {turned_bad.sum():,}")
print(f"3. ODR                        : {turned_bad.sum()/good.sum():.2%}")

1. Good in 2021-01-31          : 852
2. …that turned Bad by 2022-01-31 : 90
3. ODR                        : 10.56%


Now every observation month, rolled into the full ODR series — we chart this at the
very end to compare against the model.

In [18]:
rows = []
for i in range(len(months) - 12):
    o, e = months[i], months[i + 12]
    good = buckets[o].isin([0, 1, 2, 3])
    turned_bad = good & (buckets[e] == 4)
    if good.sum():
        rows.append({"observation": o, "window_end": e,
                     "n_good": int(good.sum()),
                     "turned_bad": int(turned_bad.sum()),
                     "odr": turned_bad.sum() / good.sum()})

odr = pd.DataFrame(rows)
odr["window_end"] = pd.to_datetime(odr["window_end"])
print(f"Observation cohorts : {len(odr)}")
print(f"Average ODR         : {odr['odr'].mean():.2%}")
print(f"Range               : {odr['odr'].min():.2%} to {odr['odr'].max():.2%}")
odr.head()

Observation cohorts : 48
Average ODR         : 7.52%
Range               : 3.87% to 14.11%


,observation,window_end,n_good,turned_bad,odr
0,2020-01-31,2021-01-31,919,77,0.083787
1,2020-02-29,2021-02-28,903,87,0.096346
2,2020-03-31,2021-03-31,912,100,0.109649
3,2020-04-30,2021-04-30,914,110,0.120350
4,2020-05-31,2021-05-31,916,125,0.136463


> **▶ SAY:** "The one thing to watch is the bottom of the fraction — it's the good
> ones at the *start*. On the real book the early-2020 cohorts run above 50% — the
> pandemic. Here it's gentler, same shape. Hold this series — we use it again at the
> end to compare against the model."

---
## 6 · Build the transition matrix

This is the core, in three moves: **Count → Percentage → Average.**

- **Count** — for one pair of months, how many accounts moved from each bucket to
  each bucket.
- **Percentage** — divide each count by its row total, so each row becomes the
  probabilities of where an account in that bucket goes next.
- **Average** — do that for every pair of adjacent months and average them, giving
  the typical, through-the-cycle behaviour.

We start with the **Count** for one month-pair — including an **Indeterminate**
column (accounts that left the book) and **Total Accounts Evaluated** (the row total
that excludes Indeterminate).

In [19]:
jan, feb = "2021-01-31", "2021-02-28"
LABELS = ["0 DPD", "1-30 DPD", "31-60 DPD", "61-90 DPD", "91+ DPD"]

def count_matrix(m_from, m_to):
    # For each starting bucket, count where those accounts ended up.
    rows = []
    for a in range(5):
        src = buckets[m_from] == a                      # accounts starting in bucket a
        row = {b: int((src & (buckets[m_to] == b)).sum()) for b in range(5)}
        row["Indeterminate"] = int((src & buckets[m_to].isna()).sum())  # left the book
        rows.append(row)
    counts = pd.DataFrame(rows, index=LABELS)
    counts["Total Accounts Evaluated"] = counts[[0, 1, 2, 3, 4]].sum(axis=1)
    return counts

count_matrix(jan, feb)

,0,1,2,3,4,Indeterminate,Total Accounts Evaluated
0 DPD,626,46,13,6,1,18,692
1-30 DPD,36,24,8,3,4,2,75
31-60 DPD,11,9,7,10,2,0,39
61-90 DPD,3,3,4,9,7,0,26
91+ DPD,0,0,0,0,95,1,95


**Percentage** = each count ÷ Total Accounts Evaluated for its row (Indeterminate
is left out of the denominator — e.g. `824 / 838 = 98.33%`). `.div(total, axis=0)`
divides each row by its own total. Each row now sums to 100%, our quick check.

In [20]:
def transition_matrix(m_from, m_to):
    counts = count_matrix(m_from, m_to)
    total = counts["Total Accounts Evaluated"]
    return counts[[0, 1, 2, 3, 4]].div(total, axis=0)   # each row / its total

one = transition_matrix(jan, feb)
print("Row sums:", one.sum(axis=1).round(4).tolist(), " <- each row = 100%")
one.style.format("{:.1%}")

Row sums: [1.0, 1.0, 1.0, 1.0, 1.0]  <- each row = 100%


,0,1,2,3,4
0 DPD,90.5%,6.6%,1.9%,0.9%,0.1%
1-30 DPD,48.0%,32.0%,10.7%,4.0%,5.3%
31-60 DPD,28.2%,23.1%,17.9%,25.6%,5.1%
61-90 DPD,11.5%,11.5%,15.4%,34.6%,26.9%
91+ DPD,0.0%,0.0%,0.0%,0.0%,100.0%


**Average** across every adjacent month-pair → the through-the-cycle matrix. We
build one grid per pair, `concat` them into a tall stack, and `groupby` the row
label and take the `mean` — which skips any period where a bucket was empty.

In [21]:
all_matrices = [transition_matrix(months[i], months[i + 1])
                for i in range(len(months) - 1)]

# Stack them, then average each row-label across all the periods.
avg = pd.concat(all_matrices).groupby(level=0).mean()
avg = avg.reindex(LABELS)                 # keep buckets in order
avg.columns = LABELS                      # label the destination columns too

print("Number of monthly grids :", len(all_matrices))
print("Row sums:", avg.sum(axis=1).round(4).tolist())

avg.index.name = "from \\ to"
avg.style.format("{:.1%}")

Number of monthly grids : 59
Row sums: [1.0, 1.0, 1.0, 1.0, 1.0]


,0 DPD,1-30 DPD,31-60 DPD,61-90 DPD,91+ DPD
from \ to,,,,,
0 DPD,94.4%,4.3%,0.7%,0.4%,0.2%
1-30 DPD,52.6%,31.1%,11.3%,3.5%,1.6%
31-60 DPD,25.5%,23.1%,27.9%,17.1%,6.5%
61-90 DPD,12.0%,15.1%,18.8%,28.6%,25.5%
91+ DPD,0.0%,0.0%,0.0%,0.0%,100.0%


> **▶ SAY:** "Three moves — count, percentages, average. On the percentage grid,
> point out every row adds to 100% (the quick check). The average `concat` +
> `groupby().mean()` does all the monthly grids at once."
> **▶ Take this one slow** — it's the core of the session.

---
## 7 · Project forward (Markov)

The matrix so far covers **one month**. We need **twelve** — so we multiply the
average matrix by itself 12 times. pandas multiplies two matrices with `.dot()`, so
we just `.dot()` in a short loop.

Why multiply at all? To go from bucket 0 to bucket 4 in two months, an account
could pass through any bucket in month one — and multiplying the matrices adds up
every one of those paths for you automatically.

In [22]:
# Multiplying two matrices in pandas is .dot(). To multiply by itself 12 times,
# we just .dot() in a loop.
def project(matrix, n):
    result = matrix.copy()
    for _ in range(n - 1):
        result = result.dot(matrix)
    return result

twelve = project(avg, 12)
twelve.style.format("{:.1%}")

,0 DPD,1-30 DPD,31-60 DPD,61-90 DPD,91+ DPD
from \ to,,,,,
0 DPD,82.8%,6.2%,2.1%,1.3%,7.6%
1-30 DPD,77.3%,5.8%,2.0%,1.2%,13.7%
31-60 DPD,66.1%,5.0%,1.7%,1.1%,26.2%
61-90 DPD,48.2%,3.6%,1.3%,0.8%,46.1%
91+ DPD,0.0%,0.0%,0.0%,0.0%,100.0%


In [23]:
# The PD building up month by month — the default column ("91+ DPD") at each step.
power, rows = avg.copy(), {}
for n in range(1, 13):
    rows[n] = power["91+ DPD"]
    power = power.dot(avg)

horizon = pd.DataFrame(rows).T
horizon.index.name = "months ahead"
horizon.style.format("{:.1%}")

from \ to,0 DPD,1-30 DPD,31-60 DPD,61-90 DPD,91+ DPD
months ahead,,,,,
1,0.2%,1.6%,6.5%,25.5%,100.0%
2,0.6%,3.8%,13.1%,34.2%,100.0%
3,1.1%,5.7%,17.0%,38.4%,100.0%
4,1.8%,7.2%,19.4%,40.6%,100.0%
5,2.4%,8.3%,20.9%,42.0%,100.0%
6,3.2%,9.3%,22.0%,43.0%,100.0%
7,3.9%,10.1%,22.9%,43.7%,100.0%
8,4.6%,10.9%,23.7%,44.3%,100.0%
9,5.4%,11.6%,24.3%,44.8%,100.0%


> **▶ SAY:** "One line does the twelve multiplications. Quick caveat you can phrase
> in your own words — it assumes each month behaves about the same. You'll explain
> that side better than me."

---
## 8 · Through-the-Cycle PD (TTC PD)

The answer is the **last column** of the 12-step matrix — the default column
(`"91+ DPD"`). That column *is* the 12-month PD for each aging bucket. This table is
the end output of the whole session.

In [24]:
ttc = pd.DataFrame({"Aging bucket": LABELS, "12-month TTC PD": twelve["91+ DPD"].values})
ttc.style.format({"12-month TTC PD": "{:.2%}"}).hide(axis="index")

Aging bucket,12-month TTC PD
0 DPD,7.56%
1-30 DPD,13.67%
31-60 DPD,26.17%
61-90 DPD,46.14%
91+ DPD,100.00%


> **▶ SAY:** "This is the deliverable — one PD per bucket, straight from the last
> column of the matrix. On the real portfolio these are 11.64, 37.83, 48.87, 75.67,
> 100. Lower here (sample data), but they climb the same way: further behind today,
> higher the PD."

---
## 9 · Compare against actual defaults

A quick reality check, no chart needed. The model's bucket-0 PD is a
*through-the-cycle average*, so it should land near the **average** of the observed
default rate we built in step 5 — sitting in the middle of the range, not tracking
any single year.

In [25]:
model_pd = twelve.loc["0 DPD", "91+ DPD"]   # model's bucket-0 12-month PD

print(f"Model bucket-0 TTC PD : {model_pd:.2%}")
print(f"Observed ODR, average : {odr['odr'].mean():.2%}")
print(f"Observed ODR, range   : {odr['odr'].min():.2%} to {odr['odr'].max():.2%}")

Model bucket-0 TTC PD : 7.56%
Observed ODR, average : 7.52%
Observed ODR, range   : 3.87% to 14.11%


> **▶ SAY:** "The model's number sits near the average of what actually happened —
> that's the check. It doesn't track any single year, because it's a
> through-the-cycle average."

---
## What you just did — the Transition Matrix, in Python

| Step | The pandas |
|---|---|
| 1 · Consolidate | `read_excel`, `concat` |
| Data validation | required columns, duplicates, types, missing % |
| 2 · Pivot | `pivot_table` |
| 3 · Bucket | a function + `.apply()` |
| 4 · Everbad | a loop carrying state (`.cummax`) |
| 5 · Default rate (ODR) | boolean masks, `.isin()`, `.sum()` |
| 6 · Build the matrix | boolean-mask counts, `.div()`, `concat` + `groupby().mean()` |
| 7 · Markov | `.dot()` in a loop |
| 8 · TTC PD | the `"91+ DPD"` column of the 12-step matrix |
| 9 · Compare vs actual | `.mean()`, `.min()`, `.max()` |

**One library, pandas — nothing else.** The hard part was never the syntax — it was
**knowing what shape the data needs to be in** at each step.

### Three things to remember
1. **Check the shape after every step.** `df.shape` catches most bugs instantly.
2. **A blank means something.** Here it's "not on the book" → the Indeterminate
   group. Decide what missing means before you handle it.
3. **Every model has one assumption that moves the answer.** Here it's everbad —
   find it, and be able to say out loud what it costs.

### Reuse it in your own work
The whole process is packaged as functions in **`transition_matrix.py`**:

```python
import transition_matrix as tm
avg, projected, ttc = tm.run_pipeline()   # point at your data via DATA_ROOT
print(ttc)                                 # 12-month TTC PD by aging bucket
```

### Point it at your real data
Set `DATA_ROOT` to the real archive (one sub-folder per year) and make sure the
four columns are named `pn_number`, `loan_segment`, `report_date`, `dpd`. On the
real portfolio the TTC PD reads **11.64% / 37.83% / 48.87% / 75.67% / 100%**.

### Where this fits
This is the **Transition Matrix** — one part of the wider ECL work. The steps that
come after it (net flow rate, ECL, and the rest) are covered in the other sessions
in this series.

> **▶ Close (in your own words):** "So that's the whole Transition Matrix, start to
> finish, in something you can read top to bottom. It's only a handful of steps, and
> every judgement call ends up as one visible block of code."
>
> **▶ Hand off:** "This is where my part stops. The pieces after this — net flow
> rate, ECL, and the rest — my colleagues will take you through in their sessions."
>
> **▶ Q&A likely to come up:**
> - *"Why 7.6% here, 11.64% on ours?"* Different data — sample vs real. Same method;
>   the real 2020 stress was sharper (point at the last chart).
> - *"Can it read our real files?"* Yes — set `DATA_ROOT`, keep the 4 column names.
> - *"Segment-level matrices?"* Filter `panel`/`buckets` by `loan_segment` first,
>   then the same steps. Happy to build it as a follow-up.